### Instruction Finetuning

In [ ]:
import json
import os
import requests

In [5]:
from importlib.metadata import version

pkgs = [
    "numpy",
    "matplotlib",
    "tiktoken",
    "torch",
    "tqdm",
    "tensorflow"
]
for p in pkgs:
    print(f"{p} version: {version(p)}")

numpy version: 2.3.4
matplotlib version: 3.10.7
tiktoken version: 0.14.0
torch version: 2.11.0
tqdm version: 4.67.1
tensorflow version: 2.21.0


### Step-1:-  Preparing the dataset

We are using Alpaca Format, every models have their own format, so the format of the tuning can differ

In [ ]:
def download_and_load_file(file_path, url):
    if not os.path.exists(file_path):
        response = requests.get(url, timeout=30)
        response.raise_for_status()
        text_data = response.text
        with open(file_path, "w", encoding="utf-8") as file:
            file.write(text_data)

    with open(file_path, "r", encoding="utf-8") as file:
        data = json.load(file)

    return data

In [8]:
file_path = "instruction-data.json"
url = (
    "https://raw.githubusercontent.com/rasbt/LLMs-from-scratch"
    "/main/ch07/01_main-chapter-code/instruction-data.json"
)

data = download_and_load_file(file_path, url)
print("Number of entries:", len(data))

Number of entries: 1100


In [ ]:
## THis format is called ALPACA format... There are different formats that exist and the data format doesn't matter, we need to take prompt style templates for instruction tuning. We need to apply those prompt templates for instruction tuning. 
data[50] , data[999]

({'instruction': 'Identify the correct spelling of the following word.',
  'input': 'Ocassion',
  'output': "The correct spelling is 'Occasion.'"},
 {'instruction': "What is an antonym of 'complicated'?",
  'input': '',
  'output': "An antonym of 'complicated' is 'simple'."})

In [15]:
def format_input(entry):
    instruction_text =(
        f"Below is an instruction that describes a task. "
        f"Write a response that appropriately completes the request."
        f"\n\n### Instruction:\n{entry['instruction']}"
    ) ## Shorter helps in token saving
    
    input_text =  f"\n\n #### Input:\n{entry['input']}" if entry['input'] else ""
    
    return instruction_text + input_text

## Example of format
print(format_input(data[50]))
print()
print(format_input(data[999]))

Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
Identify the correct spelling of the following word.

 #### Input:
Ocassion

Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
What is an antonym of 'complicated'?


In [14]:
desired_response = f"\n\n### Response:\n{data[50]['output']}"
model_input = format_input(data[50])

print(model_input + desired_response)

Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
Identify the correct spelling of the following word.

 #### Input:
Ocassion

### Response:
The correct spelling is 'Occasion.'


#### Spliting the data 

In [17]:
train_portion = int(len(data) * 0.85)  # 85% for training
test_portion = int(len(data) * 0.1)    # 10% for testing
val_portion = len(data) - train_portion - test_portion  # Remaining 5% for validation

train_data = data[:train_portion]
test_data = data[train_portion:train_portion + test_portion]
val_data = data[train_portion + test_portion:]

print("Training set length:", len(train_data))
print("Validation set length:", len(val_data))
print("Test set length:", len(test_data))

Training set length: 935
Validation set length: 55
Test set length: 110


### Step-2: - Prepare the Training Set Batches

In [18]:
import torch
from torch.utils.data import Dataset

In [19]:
from typing import Any


class InstructionDataset(Dataset):
    
    def __init__(self, ds, tokenizer) -> None:
        super().__init__()
        self.data = ds
        
        
        ## Here we are padding per batch, to save on tokens; as every batch can be very long. 
        # So 1 batch has max length 10 tokens, other can be 20 post padding
        self.encoded_texts = []
        for entry in data:
            instruction_plus_input = format_input(entry)
            response_text = f"\n\n### Response:\n{data[50]['output']}"
            full_text = instruction_plus_input + response_text
            self.encoded_texts.append(
                tokenizer.encode(full_text)
            )
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, index) -> Any:
        return self.encoded_texts[index]        

#### Setting up Tokenizer

In [21]:
import tiktoken
tokenizer = tiktoken.get_encoding("gpt2")

In [22]:
print(tokenizer.encode('<|endoftext|>', allowed_special={"<|endoftext|>"}))

[50256]
